# Test notebook

The purpose of this notebook is to test an equation and compare them with the baselines: Burton, MBR, and DDM1, 2 and 3. We will also plot each storm and get the metrics for the equation.

The only cell that we have to modify is the following one, where we can change the features, the mode (template or default) and the output directory for the plots.
Raw EQ is the equation that we want to test, the raw version generated from the train_script.py file.

In [1]:
import os

FEATURES = ["Vp", "Np", "Bzsouth", "Bmag", "DST"]
MODE = "default"  # 'template' or 'default'
OUTPUT_DIR = "test_plots_notebook_default_primitive"
RAW_EQ = "((DST - Np) * -0.04895255) + (Bzsouth * ((((DST * 2.4618247) + Vp) + (Bzsouth * Np)) * -0.0025749283))"

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [2]:
%cd /mnt/data/symbolic-regression-dst-public-repo

/mnt/data/symbolic-regression-dst-public-repo


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from tqdm import tqdm

from sympy.printing import latex

# Internal module imports
import storm_dates
import baseline_models

# from evaluation_engine import UnifiedModel, simulate_storm, compute_features
from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

[juliapkg] Found dependencies: /mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/juliacall/juliapkg.json
[juliapkg] Found dependencies: /mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/pysr/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] Using Julia 1.11.9 at /mnt/data/symbolic-regression-dst-public-repo/.venv/julia_env/pyjuliapkg/install/bin/julia
[juliapkg] Using Julia project at /mnt/data/symbolic-regression-dst-public-repo/.venv/julia_env
[juliapkg] Writing Project.toml:
           | [deps]
           | PythonCall = "6099a3de-0909-46bc-b1f4-468b9a2dfc0d"
           | OpenSSL_jll = "458c3c95-2e84-50aa-8efc-19380b2a3a95"
           | SymbolicRegression = "8254be44-1295-4e6a-a16d-46

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
    Updating `/mnt/data/symbolic-regression-ldi-internal/.venv/julia_env/Project.toml`
⌅ [6099a3de] + PythonCall v0.9.26
⌅ [8254be44] + SymbolicRegression v1.11.3
⌅ [458c3c95] + OpenSSL_jll v3.0.20+0
  [9e88b42a] ~ Serialization ⇒ v1.11.0
    Updating `/mnt/data/symbolic-regression-ldi-internal/.venv/julia_env/Manifest.toml`
  [47edcb42] + ADTypes v1.21.0
  [79e6a3ab] + Adapt v4.5.2
  [66dad0bd] + AliasTables v1.1.3
  [4fba245c] + ArrayInterface v7.24.0
  [d360d2e6] + ChainRulesCore v1.26.1
  [bbf7d656] + CommonSubexpressions v0.3.1
  [34da2185] + Compat v4.18.1
  [992eb4ea] + CondaPkg v0.2.34
  [187b0558] + ConstructionBase v1.6.0
  [9a962f9c] + DataAPI v1.16.0
  [864edb3b] + DataStructures v0.19.4
  [e2d170a0] + DataValueInterfaces v1.0.0
  [163ba53b] + DiffResults v1.1.0
  [b552c78f] + DiffRules v1.15.1
  [a0c0ee7d] + DifferentiationInterface v0.7.16
  [8d63f2c5] + DispatchDoctor v0.4.28
  [

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [4]:
raw_data = load_and_preprocess()
data = compute_features(raw_data)

Reading from file ./data/all_timeline/ace_imf_1h_1998.csv
Reading from file ./data/all_timeline/ace_imf_1h_1999.csv
Reading from file ./data/all_timeline/ace_imf_1h_2000.csv
Reading from file ./data/all_timeline/ace_imf_1h_2001.csv
Reading from file ./data/all_timeline/ace_imf_1h_2002.csv
Reading from file ./data/all_timeline/ace_imf_1h_2003.csv
Reading from file ./data/all_timeline/ace_imf_1h_2004.csv
Reading from file ./data/all_timeline/ace_imf_1h_2005.csv
Reading from file ./data/all_timeline/ace_imf_1h_2006.csv
Reading from file ./data/all_timeline/ace_imf_1h_2007.csv
Reading from file ./data/all_timeline/ace_imf_1h_2008.csv
Reading from file ./data/all_timeline/ace_imf_1h_2009.csv
Reading from file ./data/all_timeline/ace_imf_1h_2010.csv
Reading from file ./data/all_timeline/ace_imf_1h_2011.csv
Reading from file ./data/all_timeline/ace_imf_1h_2012.csv
Reading from file ./data/all_timeline/ace_imf_1h_2013.csv
Reading from file ./data/all_timeline/ace_imf_1h_2014.csv
Reading from f

In [5]:
def predict_and_plot_storm(model, start, end, storm_df, storm_id, save_path):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values

    res_eq = simulate_storm(model, storm_df)
    res_burton = baseline_models.burton_prediction(storm_df)
    res_obm = baseline_models.obm_prediction(storm_df)
    ddm1 = baseline_models.ddm1_prediction(storm_df)
    ddm2 = baseline_models.ddm2_prediction(storm_df)
    ddm3 = baseline_models.ddm3_prediction(storm_df)

    res_eq = res_eq[start:end]["DST_pred"].values
    res_burton = res_burton[start:end]["DST_pred"].values
    res_obm = res_obm[start:end]["DST_pred"].values
    res_ddm1 = ddm1[start:end]["DST_pred"].values
    res_ddm2 = ddm2[start:end]["DST_pred"].values
    res_ddm3 = ddm3[start:end]["DST_pred"].values

    # 2. Calculate Metrics
    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    # 3. Setup Figure (3 Columns)
    fig, axs = plt.subplots(1, 3, figsize=(24, 7), constrained_layout=True)
    fig.suptitle(
        rf"Evaluation for Equation: ${model.latex_str()}$", fontsize=16, wrap=True
    )
    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_eq,
        color="blue",
        linestyle="--",
        label="Equation",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_burton,
        color="yellow",
        linestyle="--",
        label="Burton",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_obm,
        color="green",
        linestyle="--",
        label="OBM",
        linewidth=1.5,
    )
    
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm1,
        color="orange",
        linestyle="--",
        label="DDM1",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm2,
        color="purple",
        linestyle="--",
        label="DDM2",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm3,
        color="cyan",
        linestyle="--",
        label="DDM3",
        linewidth=1.5,
    )
    
    axs[0].set_title(f"Storm {storm_id} Reconstruction")
    axs[0].legend()
    axs[0].grid(True)
    axs[0].set_xlim(start, end)

    # Column 2: Prediction Error
    diff_eq = res_eq - y_true
    diff_burton = res_burton - y_true
    diff_obm = res_obm - y_true
    diff_ddm1 = res_ddm1 - y_true
    diff_ddm2 = res_ddm2 - y_true
    diff_ddm3 = res_ddm3 - y_true


    axs[1].plot(storm_df[start:end].index, diff_eq, color="blue", label="Eq Error")
    axs[1].plot(
        storm_df[start:end].index, diff_burton, color="yellow", label="Burton Error"
    )
    axs[1].plot(storm_df[start:end].index, diff_obm, color="green", label="OBM Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm1, color="orange", label="DDM1 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm2, color="purple", label="DDM2 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm3, color="cyan", label="DDM3 Error")
    axs[1].axhline(0, color="black", linestyle="--")

    title_metrics = (
        f"Error Comparison\n"
        f"Eq: RMSE {m_eq[0]:.2f} | MAE {m_eq[1]:.2f} | R2 {m_eq[2]:.2f} | CC {m_eq[3]:.2f} | BFE {m_eq[4]:.2f}\n"
        f"Burton: RMSE {m_burton[0]:.2f} | MAE {m_burton[1]:.2f} | R2 {m_burton[2]:.2f} | CC {m_burton[3]:.2f} | BFE {m_burton[4]:.2f}\n"
        f"OBM: RMSE {m_obm[0]:.2f} | MAE {m_obm[1]:.2f} | R2 {m_obm[2]:.2f} | CC {m_obm[3]:.2f} | BFE {m_obm[4]:.2f}\n"
        f"DDM1: RMSE {m_ddm1[0]:.2f} | MAE {m_ddm1[1]:.2f} | R2 {m_ddm1[2]:.2f} | CC {m_ddm1[3]:.2f} | BFE {m_ddm1[4]:.2f}\n"
        f"DDM2: RMSE {m_ddm2[0]:.2f} | MAE {m_ddm2[1]:.2f} | R2 {m_ddm2[2]:.2f} | CC {m_ddm2[3]:.2f} | BFE {m_ddm2[4]:.2f}\n"
        f"DDM3: RMSE {m_ddm3[0]:.2f} | MAE {m_ddm3[1]:.2f} | R2 {m_ddm3[2]:.2f} | CC {m_ddm3[3]:.2f} | BFE {m_ddm3[4]:.2f}"
    )
    axs[1].set_title(title_metrics, fontsize=9)
    axs[1].set_ylabel("Error (nT)")
    axs[1].legend()
    axs[1].grid(True)
    axs[1].set_xlim(start, end)

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        [res_eq, res_burton, res_obm, res_ddm1, res_ddm2, res_ddm3],
        ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"],
        ["blue", "yellow", "green", "orange", "purple", "cyan"],
    )

    plt.savefig(save_path)
    plt.close()


In [6]:
def save_prediction_data(model, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eq = simulate_storm(model, storm_df)
    if model.is_template:
        pred_dst_eq = pred_dst_eq[start:end][
            ["DST_pred", "dDST", "injection_component", "decay_component"]
        ]
    else:
        pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
    # 3. Baseline Predictions (Burton & OBM)
    pred_dst_burton = baseline_models.burton_prediction(storm_df)
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred", "dDST"]]
    pred_dst_burton.columns = ["DST_pred_burton", "dDST_burton"]
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred_burton", "dDST_burton"]]
    pred_dst_obm = baseline_models.obm_prediction(storm_df)
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred", "dDST"]]
    pred_dst_obm.columns = ["DST_pred_obm", "dDST_obm"]
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred_obm", "dDST_obm"]]
    pred_dst_ddm1 = baseline_models.ddm1_prediction(storm_df)    
    pred_dst_ddm1.columns = ["DST_pred_ddm1", "dDST_ddm1"]
    pred_dst_ddm1 = pred_dst_ddm1[start:end][["DST_pred_ddm1", "dDST_ddm1"]]
    pred_dst_ddm2 = baseline_models.ddm2_prediction(storm_df)
    pred_dst_ddm2.columns = ["DST_pred_ddm2", "dDST_ddm2"]
    pred_dst_ddm2 = pred_dst_ddm2[start:end][["DST_pred_ddm2", "dDST_ddm2"]]
    pred_dst_ddm3 = baseline_models.ddm3_prediction(storm_df)
    pred_dst_ddm3.columns = ["DST_pred_ddm3", "dDST_ddm3"]
    pred_dst_ddm3 = pred_dst_ddm3[start:end][["DST_pred_ddm3", "dDST_ddm3"]]


    # 4. Construct Comprehensive DataFrame

    if model.is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,
            }
        ).set_index("Timestamp")
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,

            }
        ).set_index("Timestamp")

    results_df.to_csv(output_path)
    return results_df

## Test storms

In [7]:
storms = []

model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
test_storms = storm_dates.TEST_STORMS_SYMBOLIC_REGRESSION
storm_indices = []
for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        model,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            model, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)
    
with open(os.path.join(OUTPUT_DIR, 'equation.txt'), 'w') as f:
    f.write(f'Equation: {RAW_EQ}\n')            
    f.write(f'LaTeX: {latex(model.latex_str())}\n')

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:12<00:00,  1.57it/s]


In [8]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    res_eq = storm["Pred_DST_Equation"].values
    res_burton = storm["Pred_DST_Burton"].values
    res_obm = storm["Pred_DST_OBM"].values
    res_ddm1 = storm["Pred_DST_DDM1"].values
    res_ddm2 = storm["Pred_DST_DDM2"].values
    res_ddm3 = storm["Pred_DST_DDM3"].values

    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    summary_df.loc[len(summary_df)] = [
        storm_indices[storm_index],
        *m_eq,
        *m_burton,
        *m_obm,
        *m_ddm1,
        *m_ddm2,
        *m_ddm3,
    ]

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values
res_eq = global_data["Pred_DST_Equation"].values
res_burton = global_data["Pred_DST_Burton"].values
res_obm = global_data["Pred_DST_OBM"].values
res_ddm1 = global_data["Pred_DST_DDM1"].values
res_ddm2 = global_data["Pred_DST_DDM2"].values
res_ddm3 = global_data["Pred_DST_DDM3"].values

m_eq = baseline_models.get_all_metrics(y_true, res_eq)
m_burton = baseline_models.get_all_metrics(y_true, res_burton)
m_obm = baseline_models.get_all_metrics(y_true, res_obm)
m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

summary_df.loc[len(summary_df)] = [
    "Global",
    *m_eq,
    *m_burton,
    *m_obm,
    *m_ddm1,
    *m_ddm2,
    *m_ddm3,
]

display(summary_df)

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_CC,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_CC,...,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_CC,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_CC,DDM3_BFE
0,90.0,12.200250,9.334337,0.632634,0.936198,15.866722,12.546739,9.426567,0.611471,0.804031,...,17.034353,14.718489,0.283836,0.819670,19.285046,17.429228,14.902320,0.250248,0.849179,20.944809
1,91.0,16.101688,13.040237,0.785731,0.908660,16.417254,23.402983,17.430162,0.547353,0.869635,...,20.994874,17.672795,0.635713,0.870809,22.088199,22.442717,18.173790,0.583737,0.864456,25.386413
2,92.0,11.068902,8.716594,0.731048,0.876615,13.648058,22.444927,16.827487,-0.105866,0.856593,...,11.710892,9.329760,0.698945,0.874185,13.104050,11.378825,9.160091,0.715776,0.872556,12.828221
3,93.0,10.061079,8.250151,0.742653,0.880514,8.561342,19.062180,15.620942,0.076206,0.792714,...,8.303250,6.470139,0.824723,0.917542,11.441876,8.670891,6.332414,0.808857,0.904447,13.883947
4,94.0,8.466631,6.227716,0.826793,0.926282,13.938754,17.059837,14.234517,0.296775,0.867234,...,11.990044,9.586551,0.652635,0.926882,19.031650,11.862568,8.934847,0.659982,0.920503,19.970899
5,95.0,14.562744,12.563000,0.786442,0.952266,11.121084,27.156568,23.604603,0.257358,0.894657,...,10.840170,8.734074,0.881668,0.955437,13.497036,12.354960,9.578556,0.846286,0.954712,16.759531
6,96.0,18.325205,14.755451,0.759516,0.936895,19.979894,31.161847,25.739701,0.304599,0.930523,...,13.320289,10.330457,0.872938,0.963738,17.936865,14.717364,11.742180,0.844887,0.954268,19.772733
7,97.0,11.826668,8.773574,0.916614,0.964248,16.328690,43.516911,23.133641,-0.128974,0.898528,...,22.490189,16.826801,0.698454,0.883757,30.616512,20.468681,15.214925,0.750226,0.886201,30.187252
8,98.0,13.456650,11.220634,0.579011,0.850906,12.298406,24.764109,20.315994,-0.425746,0.715543,...,10.405858,7.936314,0.748260,0.888742,14.058279,11.805307,9.352887,0.675995,0.859124,16.881751
9,99.0,16.558724,13.501615,0.800790,0.914560,21.085079,16.754713,13.386004,0.796046,0.900956,...,20.014049,16.071049,0.708977,0.932586,28.117249,22.085126,17.377477,0.645630,0.929705,32.083276


In [9]:
display(summary_df[['Storm Index', 'Equation_BFE', 'Burton_BFE', 'OBM_BFE', 'DDM1_BFE', 'DDM2_BFE', 'DDM3_BFE']])

,Storm Index,Equation_BFE,Burton_BFE,OBM_BFE,DDM1_BFE,DDM2_BFE,DDM3_BFE
0,90.0,15.866722,16.120563,23.690554,21.129540,19.285046,20.944809
1,91.0,16.417254,31.626866,22.218178,23.435284,22.088199,25.386413
2,92.0,13.648058,33.054183,16.444423,13.923395,13.104050,12.828221
3,93.0,8.561342,21.054172,13.763652,13.120449,11.441876,13.883947
4,94.0,13.938754,13.099165,15.762979,20.108684,19.031650,19.970899
5,95.0,11.121084,28.255954,19.869271,14.982619,13.497036,16.759531
6,96.0,19.979894,38.832986,24.569318,36.468144,17.936865,19.772733
7,97.0,16.328690,67.308999,24.519791,28.911696,30.616512,30.187252
8,98.0,12.298406,25.336156,19.973342,14.844234,14.058279,16.881751
9,99.0,21.085079,17.360182,34.541056,29.948383,28.117249,32.083276


## Train storms

In [10]:
storms = []

train_storms = storm_dates.TRAIN_STORMS_SYMBOLIC_REGRESSION
storm_indices = []
for sd, ed, storm_id in tqdm(train_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        print(f'Storm {storm_id} has no data from {start} to {end}. Skipping.')
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        model,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            model, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)

100%|██████████| 53/53 [00:34<00:00,  1.56it/s]


In [11]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    res_eq = storm["Pred_DST_Equation"].values
    res_burton = storm["Pred_DST_Burton"].values
    res_obm = storm["Pred_DST_OBM"].values
    res_ddm1 = storm["Pred_DST_DDM1"].values
    res_ddm2 = storm["Pred_DST_DDM2"].values
    res_ddm3 = storm["Pred_DST_DDM3"].values

    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    summary_df.loc[len(summary_df)] = [
        storm_indices[storm_index],
        *m_eq,
        *m_burton,
        *m_obm,
        *m_ddm1,
        *m_ddm2,
        *m_ddm3,
    ]

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values
res_eq = global_data["Pred_DST_Equation"].values
res_burton = global_data["Pred_DST_Burton"].values
res_obm = global_data["Pred_DST_OBM"].values
res_ddm1 = global_data["Pred_DST_DDM1"].values
res_ddm2 = global_data["Pred_DST_DDM2"].values
res_ddm3 = global_data["Pred_DST_DDM3"].values

m_eq = baseline_models.get_all_metrics(y_true, res_eq)
m_burton = baseline_models.get_all_metrics(y_true, res_burton)
m_obm = baseline_models.get_all_metrics(y_true, res_obm)
m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

summary_df.loc[len(summary_df)] = [
    "Global",
    *m_eq,
    *m_burton,
    *m_obm,
    *m_ddm1,
    *m_ddm2,
    *m_ddm3,
]

display(summary_df)

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_CC,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_CC,...,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_CC,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_CC,DDM3_BFE
0,2.0,16.031464,13.730000,0.773782,0.911157,12.396209,31.484362,20.573715,0.127490,0.818256,...,22.568521,19.475600,0.551681,0.896125,14.829997,23.252721,19.882235,0.524086,0.889651,19.966095
1,3.0,15.908807,12.515820,0.350755,0.673083,15.619399,23.937685,19.459180,-0.469933,0.761242,...,18.874640,14.104114,0.086117,0.588612,18.701587,21.678056,16.169453,-0.205519,0.501767,21.927198
2,4.0,12.524202,9.529419,0.849204,0.929801,13.192964,21.712222,18.382958,0.546791,0.819720,...,14.543500,12.109214,0.796658,0.933442,19.227867,15.653912,12.848885,0.764421,0.924793,21.447613
3,5.0,15.218147,12.666556,0.826682,0.919367,18.935738,44.853190,27.594408,-0.505591,0.860603,...,20.949735,17.795732,0.671544,0.887207,21.903819,20.551589,17.276006,0.683910,0.878793,22.014097
4,6.0,20.378345,15.070956,0.722153,0.867898,26.766279,35.150988,19.700443,0.173309,0.896534,...,22.171150,16.574360,0.671115,0.855711,29.435293,22.597948,16.987666,0.658331,0.843171,29.757253
5,7.0,12.161474,9.718818,0.782593,0.915574,13.449344,32.374648,18.412785,-0.540676,0.840241,...,18.066086,15.496299,0.520234,0.820916,17.225424,17.936579,15.021498,0.527088,0.822802,17.003694
6,8.0,12.051644,10.678982,0.746266,0.877783,9.168755,15.301777,11.774139,0.590957,0.834094,...,12.238735,10.768601,0.738327,0.915610,14.937157,13.896036,12.057634,0.662661,0.897081,18.509792
7,9.0,11.067485,8.751505,0.875407,0.959191,10.693658,27.733494,19.414167,0.217643,0.910565,...,10.379121,8.237262,0.890424,0.959182,8.695989,11.590607,9.317260,0.863351,0.957484,10.319187
8,10.0,9.713326,6.646768,0.816737,0.926262,8.121110,17.988147,13.323236,0.371491,0.863801,...,11.275480,8.936104,0.753050,0.884648,10.333607,12.823987,9.634174,0.680563,0.852321,11.559362
9,11.0,10.773170,7.077217,0.754091,0.878465,14.604487,23.923211,18.938189,-0.212625,0.808560,...,14.126297,8.265998,0.577191,0.811954,21.448999,16.740966,8.877678,0.406188,0.721946,25.359561


In [12]:
display(summary_df[['Storm Index', 'Equation_BFE', 'Burton_BFE', 'OBM_BFE', 'DDM1_BFE', 'DDM2_BFE', 'DDM3_BFE']])

,Storm Index,Equation_BFE,Burton_BFE,OBM_BFE,DDM1_BFE,DDM2_BFE,DDM3_BFE
0,2.0,12.396209,38.924063,22.972515,13.437801,14.829997,19.966095
1,3.0,15.619399,24.860282,16.204448,19.256149,18.701587,21.927198
2,4.0,13.192964,19.457216,24.581542,20.804152,19.227867,21.447613
3,5.0,18.935738,53.421972,20.592857,21.296023,21.903819,22.014097
4,6.0,26.766279,60.754601,15.980338,27.490935,29.435293,29.757253
5,7.0,13.449344,40.312244,17.374153,12.948621,17.225424,17.003694
6,8.0,9.168755,11.754159,20.473563,17.712507,14.937157,18.509792
7,9.0,10.693658,35.166427,20.710150,9.832557,8.695989,10.319187
8,10.0,8.121110,19.489074,14.702154,11.832959,10.333607,11.559362
9,11.0,14.604487,39.337266,27.613516,19.511731,21.448999,25.359561
